# A/B Test Basics: ITT vs As-Treated (Beginner-Friendly)

This notebook walks through a simple A/B test analysis using **pandas** and **matplotlib**.

**Columns in the dataset**
- `assigned_group`: A (control) or B (treatment), assigned at random
- `participated`: whether the user actually participated (shows assignment → participation drop-off)
- `received_treatment`: whether the user actually received treatment (noncompliance)
- `purchase`: purchase outcome (some values are missing to mimic attrition)

> The dataset is **synthetic** (simulated) and contains **no proprietary or personal data**.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../data/synthetic_ab_test_beginner.csv"
df = pd.read_csv(DATA_PATH)

df.head()

## 1) Basic counts

We start by checking how many users were assigned to each group.

In [ ]:
df['assigned_group'].value_counts()

## 2) Participation rate (assignment → participation drop-off)

In real experiments, not everyone who is assigned to a group follows through.  
This is one reason why **ITT** analysis is used: it sticks to assignment.

In [ ]:
participation_rate = df.groupby('assigned_group')['participated'].mean().sort_index()
participation_rate

In [ ]:
participation_rate.plot(kind='bar')
plt.ylim(0, 1)
plt.ylabel("Participation rate")
plt.title("Participation rate by assigned group")
plt.tight_layout()
plt.show()

## 3) Missing outcomes (attrition)

Some purchases are missing (`NaN`), which can happen if outcomes are not observed for everyone.  
We report the missing rate by group.

In [ ]:
missing_rate = df.groupby('assigned_group')['purchase'].apply(lambda s: s.isna().mean()).sort_index()
missing_rate

## 4) ITT estimate (by assigned group)

**ITT (Intention-to-Treat)** compares outcomes by the group people were assigned to.  
Here we compute purchase rates using **observed outcomes only** (complete-case analysis).

In [ ]:
df_obs = df.dropna(subset=['purchase'])
itt_rate = df_obs.groupby('assigned_group')['purchase'].mean().sort_index()
itt_rate

In [ ]:
itt_rate.plot(kind='bar')
plt.ylabel("Observed purchase rate")
plt.title("ITT: Purchase rate by assigned group (observed outcomes only)")
plt.tight_layout()
plt.show()

## 5) As-treated estimate (by treatment received)

The **as-treated** view compares people by whether they actually received treatment.  
This can be informative, but it can also be biased if receiving treatment is related to user characteristics.

We compute purchase rates by `received_treatment` (0/1).

In [ ]:
as_treated_rate = df_obs.groupby('received_treatment')['purchase'].mean().sort_index()
as_treated_rate

In [ ]:
as_treated_rate.plot(kind='bar')
plt.ylabel("Observed purchase rate")
plt.title("As-treated: Purchase rate by treatment received")
plt.tight_layout()
plt.show()

## 6) Takeaways (high level)

- **Noncompliance** tends to make ITT effects smaller (diluted) than as-treated effects.
- **Missing outcomes** can distort conclusions if missingness differs by group.
- In real applications, strong conclusions require careful design and transparent reporting.
